# Bot setup

The Discord bot and the Supabase database are the front end. The bot is a separate distribution, `tri-bot` (`bot/pyproject.toml`), that depends on `tri` and imports only `tri.api`. All network access lives in the bot: Discord, Supabase, dwt.sh, deepwoken.co and pocamind/data. The bot draws nothing; the graphs are on the project site.

```
pip install . ./bot
cp .env.example .env        # fill in; never commit .env
python -m bot
```

Required: `DISCORD_TOKEN`, `SUPABASE_URL`, `SUPABASE_SERVICE_KEY`. Server roles and channels are set in Discord with `/setup`. Every other variable, with its default, is listed in `.env.example` and in the `bot/config.py` docstring; the next cell prints them.

In [ ]:
import bot.config as c
print(c.__doc__)

## Runs

```
python -m bot.gamesource fetch runs/gamedata        # download pocamind/data
python -m tri.cli.fit --builds builds.json --out runs --name live
python -m bot.runstore publish runs/live            # upload to Supabase Storage
python -m bot.runstore fetch runs --name live
```

On start the bot loads the run, reads reviews, locators, authored rules, admin values and categories from the tables and hands them to the library. A new review takes effect at once: the bot builds the next state on a fork (`with_memory`) and swaps it in under its lock while answers keep running on the current state.

## Hosting

`render.yaml` defines one Render background worker (`plan: 1c-2g`) with a 5 GB disk at `/var/data`, `TRI_RUN=/var/data/runs` and `TRI_RUN_REMOTE=latest`. `.python-version` pins Python 3.12 for every Render service, including ones not created from the Blueprint.

## The graph site

The concept space, the archetype atlas, the worth space and the relative worth space are static pages on GitHub Pages, one tab each, at `https://<owner>.github.io/<repo>/` (for this repository [flanface.github.io/tri](https://flanface.github.io/tri/)). Each page turns, zooms and pans; its command box finds a build, item, locator or centre by name, and on the worth tabs a name adds the item's vector $v_i - o$ from the dense origin $o$.

```
python -m bot.site build --run runs/live                      # reads the tables with .env
python -m bot.site build --run runs/live --snapshot snap.json
git add site && git commit -m "site" && git push
```

`.github/workflows/pages.yml` publishes `site/` whenever it changes on `main`, with the workflow's own token: no key. Once, in the repository: Settings → Pages → Source: GitHub Actions. Rebuild after a refit or when the locators change; the site is public, so it shows build ids, locator labels and archetype names.

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable, "-m", "bot.site", "build", "--help"], capture_output=True, text=True).stdout)